# 01 — Type Hints, the Useful Parts

Python is dynamically typed at runtime — passing wrong types doesn't raise on its own. Type **hints** are annotations that:

1. Document intent in code (no separate doc to drift).
2. Power editor autocomplete and refactoring.
3. Let a *type checker* (mypy / pyright) flag bugs **before** you run the code.

Hints don't change runtime behavior. They are evaluated as expressions when the module loads, but Python doesn't enforce them.

## Built-in collection generics — `list[int]`, `dict[str, T]`

Since 3.9 you parameterize the built-in types directly. The old `typing.List[int]` form still works but is unnecessary.

In [1]:
def average(xs: list[float]) -> float:
    return sum(xs) / len(xs)

scores: dict[str, list[int]] = {
    "alice": [90, 85, 92],
    "bob":   [70, 75, 80],
}

print(average(scores["alice"]))
print({name: average(s) for name, s in scores.items()})

89.0
{'alice': 89.0, 'bob': 75.0}


## `Optional` and `None` — `X | None`

Many functions return *something or nothing*. Spell that as `X | None` (3.10+) — clearer than the older `Optional[X]`.

`Optional[X]` is **literally** `X | None`. They mean the same thing.

In [2]:
def find_user(name: str) -> dict | None:
    db = {"alice": {"id": 1}, "bob": {"id": 2}}
    return db.get(name)

user = find_user("alice")
if user is not None:                 # narrow the type — mypy understands this
    print(user["id"])
else:
    print("not found")

1


## Unions — `int | str`

Either-or types. The function/caller has to handle each case.

In [3]:
def normalize(value: int | str) -> int:
    # mypy will force us to handle BOTH branches.
    if isinstance(value, str):
        return int(value)
    return value

print(normalize(42))
print(normalize("42"))

42
42


## `Any` — the escape hatch

`Any` means "I'm not telling mypy anything about this." It silences type errors but eliminates the value of typing.

Reach for `Any` only at boundaries you really can't type — wild dicts from a JSON blob, raw input from a user. Prefer `object` (which forces casts) when you can.

In [4]:
from typing import Any

def parse_config(raw: Any) -> dict[str, Any]:
    # raw could be anything — from a YAML/JSON load.
    if not isinstance(raw, dict):
        raise TypeError("expected a mapping at top level")
    return raw

print(parse_config({"host": "localhost", "port": 8080}))

{'host': 'localhost', 'port': 8080}


## Callables — `Callable[[int, str], bool]`

A callable type spells its arg types and return type. Use it when a function takes another function.

In [5]:
from collections.abc import Callable     # since 3.9, prefer collections.abc over typing for Callable

def transform_all(items: list[str], fn: Callable[[str], str]) -> list[str]:
    return [fn(x) for x in items]

print(transform_all(["a", "b", "c"], str.upper))
print(transform_all([" x ", " y "], str.strip))
print(transform_all(["hello", "world"], lambda s: s[::-1]))

['A', 'B', 'C']
['x', 'y']
['olleh', 'dlrow']


## Literals — exact values as types

`Literal` lets you say "this string can ONLY be 'r' or 'w'" — useful for flags and modes.

In [6]:
from typing import Literal

Mode = Literal["r", "w", "a"]

def open_file(path: str, mode: Mode) -> str:
    return f"opening {path} in {mode!r} mode"

print(open_file("data.txt", "r"))
# print(open_file("data.txt", "x"))   # mypy: error — "x" is not in Literal["r","w","a"]

opening data.txt in 'r' mode


## `TypedDict` — typed dicts when you can't use a dataclass

Sometimes you're stuck with a dict shape (JSON in, JSON out). `TypedDict` lets mypy check it.

In [7]:
from typing import TypedDict

class UserDict(TypedDict):
    name: str
    age: int
    email: str

u: UserDict = {"name": "Alice", "age": 30, "email": "a@b.c"}
print(u["name"])

# At runtime, UserDict IS just a dict — no validation.
print(isinstance(u, dict))
print(type(u).__name__)

Alice
True
dict


## Hints are evaluated lazily for forward references

If a class references itself in its own annotations, you'd hit a `NameError` because the class isn't done being defined. The fix: `from __future__ import annotations` at the top of the file, which makes ALL annotations strings at runtime (evaluated by mypy / IDE only).

We've been using this in the `.py` files in earlier folders. It's the modern default.

In [8]:
from __future__ import annotations    # MUST be the first import

class Node:
    def __init__(self, value: int, next: Node | None = None) -> None:
        # Without `from __future__ import annotations`, `Node` here would NameError
        # because `Node` isn't fully defined yet.
        self.value = value
        self.next = next

head = Node(1, Node(2, Node(3)))
while head is not None:
    print(head.value)
    head = head.next

1
2
3


## Mini-exercise

1. Annotate this function fully (including the return type):
   ```python
   def group_by_first_letter(words):
       buckets = {}
       for w in words:
           buckets.setdefault(w[0], []).append(w)
       return buckets
   ```
2. Define a `TypedDict` for a `Book` record (`title`, `author`, `pages`, optional `subtitle`). Use `NotRequired` from `typing` for the optional field.
3. Write `safe_first(xs: list[int]) -> int | None`. Why is `int | None` better than raising on empty input? Why is it better than returning `-1`?

In [9]:
# your solutions here


**Takeaways**

- Use the modern forms: `list[int]`, `dict[str, T]`, `X | None`, `int | str`.
- `Optional[X]` ≡ `X | None`. Prefer the latter.
- `Any` opts out of checking — minimize.
- `Callable[[args], ret]` for function-typed parameters.
- `Literal` and `TypedDict` for fine-grained constraints.
- `from __future__ import annotations` makes self-references work and is the modern default.

Next: [02_generics.ipynb](02_generics.ipynb) — write functions/classes that work for any type.